In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path 


# Generate conditions for azimuth spotlight test.

Target at 0, 90, or 270 aziuth    
Distractor presented at delta of 10, 20, 30, 40, 60, or 90 degrees azimuth  (clockwise is 90, counter clockwise if 270)
All signals at 0 elevation  and 0dB SNR     

### Will use minimal reverb room 
___
## This Notebook
Generate location conditions to get model performance 
___

## Use minimally reverberant simulation of MIT 46 1004

### Load manifest of rooms

In [5]:
new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')
room_manifest

,head_azim,head_pos_xyz,index_room,is_outdoor,material_x0,material_x1,material_y0,material_y1,material_z0,material_z1,room_dim_xyz,room_materials
0,0,"[2.3, 3.6, 0.9]",0,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[4.66, 5.9, 2.48]","[11, 11, 11, 11, 15, 20]"
1,0,"[3.6, 2.36, 0.9]",1,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[5.9, 4.66, 2.48]","[11, 11, 11, 11, 15, 20]"
2,0,"[2.36, 2.3, 0.9]",2,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[4.66, 5.9, 2.48]","[11, 11, 11, 11, 15, 20]"
3,0,"[2.3, 2.3, 0.9]",3,False,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,wood panelling on glass fiber blanket,carpet on foam rubber padding,"highly absorptive panels, 1in, 16in below ceiling","[5.9, 4.66, 2.48]","[11, 11, 11, 11, 15, 20]"


### Emulation of actual human experiment

Symmetric distractor is true for azimuth and elevation - we are using two distractors at same distractor location in elevation trials

In [11]:
## Get target and distractor azimuths from human stimuli generation parameters 
human_stim_dir = Path("/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024")
human_spatial_cond_dict = pickle.load(open(human_stim_dir / 'human_azim_spotlight_cond_map.pkl', 'rb'))

# fix SNR and room index to match 
snr = 0
room_ix = 0

new_min_rev_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_manifest = pd.read_pickle(new_min_rev_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (new_min_rev_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (new_min_rev_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment formatted for model scripts 

cond_dict = {ix:{'target_loc': (cond_dict['target_azim'], 0),
                      'distract_loc': [cond_dict['distractor_azim'], 0],
                      'snr': snr,
                      'symmetric_distractor': False,  # Single distractor test 
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, cond_dict in
              human_spatial_cond_dict.items()} 

# Assert keys contiguous
assert np.all(np.array(list(cond_dict.keys())) == np.arange(len(cond_dict)))
print(len(cond_dict))

27


In [14]:

with open('binaural_test_manifests/sim_human_azim_spotlight_experiment_min_reverb_mit_room.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)

# Make anechoic conditions 

### Room 0 in set of eval rooms is anechoic simulation

In [4]:
## Get target and distractor azimuths from human stimuli generation parameters 
human_stim_dir = Path("/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024")
human_spatial_cond_dict = pickle.load(open(human_stim_dir / 'human_azim_spotlight_cond_map.pkl', 'rb'))

# fix SNR and room index to match 
snr = 0
room_ix = 0

anechoic_dir = Path("/om2/user/msaddler/spatial_audio_pipeline/assets/brir/eval")
room_manifest = pd.read_pickle(anechoic_dir / 'manifest_room.pdpkl')

def get_room_str(room_ix):
    return  (new_min_rev_dir / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (new_min_rev_dir / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment formatted for model scripts 

cond_dict = {ix:{'target_loc': (cond_dict['target_azim'], 0),
                      'distract_loc': [cond_dict['distractor_azim'], 0],
                      'snr': snr,
                      'symmetric_distractor': False,  # Single distractor test 
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, cond_dict in
              human_spatial_cond_dict.items()} 

# Assert keys contiguous
assert np.all(np.array(list(cond_dict.keys())) == np.arange(len(cond_dict)))
print(len(cond_dict))

27


In [6]:
with open('binaural_test_manifests/sim_human_azim_spotlight_experiment_anechoic_room.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)

## Manifest for alternate HRTF run 

In [2]:
alt_ear_path = Path("/om/scratch/Wed/salavill/brirs/sadie/mit_bldg46room1004/sadie_h3/")

In [12]:
## Get target and distractor azimuths from human stimuli generation parameters 
human_stim_dir = Path("/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024")
human_spatial_cond_dict = pickle.load(open(human_stim_dir / 'human_azim_spotlight_cond_map.pkl', 'rb'))


# fix SNR and room index to match 
snr = 0
room_ix = 0

alt_ear_path = Path("/om/scratch/Wed/salavill/brirs/sadie/mit_bldg46room1004/sadie_h18/")
brir_manifest = pd.read_pickle(alt_ear_path / 'manifest_brir.pdpkl')

def get_room_str(room_ix):
    return  (alt_ear_path / f"room{room_ix:04}.hdf5").as_posix()

def get_room_meta_dict(room_ix):
    manifest_path = (alt_ear_path / "manifest_brir.pdpkl").as_posix()
    index_room = room_ix
    h5_fn = get_room_str(room_ix)
    return dict(room_manifest=manifest_path, index_room=index_room, h5_fn=h5_fn)

# create dict of trial dicts for azimuth experiment formatted for model scripts 

cond_dict = {ix:{'target_loc': (cond_dict['target_azim'], 0),
                      'distract_loc': [cond_dict['distractor_azim'], 0],
                      'snr': snr,
                      'symmetric_distractor': False,  # Single distractor test 
                      'test_room_meta': get_room_meta_dict(room_ix)} for ix, cond_dict in
              human_spatial_cond_dict.items()} 

# Assert keys contiguous
assert np.all(np.array(list(cond_dict.keys())) == np.arange(len(cond_dict)))
print(len(cond_dict))

27


In [13]:
with open('binaural_test_manifests/sim_human_azim_spotlight_experiment_alt_ears_sadie_h18.pkl', 'wb') as f:
    pickle.dump(cond_dict, f)